# Does `check_labels.py` work?

This notebook calls every function in `tools/check_labels.py` and checks what comes back. Each `check(...)` line stops the notebook if the result is wrong, and the last cell lists them all.

**The labels here are fake.** They are repeating patterns assigned by row position, without reading the text, so **no number below says anything about the project**. They only test the code. Real labels still come from the group.

Everything runs on copies in a temporary folder. The last section confirms the real sheets and `data/results/label_checks.json` did not change.

To run it: install `requirements.txt`, open this file, and choose Run All. It needs `labeling/*.csv` and `data/challenges.parquet`, which are both in the repo.

In [ ]:
import contextlib
import hashlib
import io
import json
import math
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd
from afinn import Afinn
from scipy.stats import binomtest

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "tools"))
import check_labels as cl

REAL_SHEETS, REAL_OUT = cl.SHEETS, cl.OUT
CHALLENGE, NAME, STANCE = "is_challenge_to_floor_yes_no", "same_person_yes_no", "stance_hostile_neutral_friendly"
FOUR_SHEETS = ["challenge_precision", "challenge_missed", "name_resolution", "challenge_stance"]
STANCE_PATTERN = ["neutral", "hostile", "neutral", "friendly"]

fingerprint = lambda: {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in [*sorted(REAL_SHEETS.glob("*.csv")), REAL_OUT]}
before = fingerprint()
workspace = tempfile.TemporaryDirectory()
checks = []


def check(what, passed):
    """Record one check, and stop the notebook if it failed."""
    checks.append({"section": section, "check": what, "result": "PASS" if passed else "FAIL"})
    assert passed, what
    print("PASS ", what)


def fresh():
    """A new folder of blank sheet copies, with the scorer pointed at it."""
    folder = Path(tempfile.mkdtemp(dir=workspace.name))
    for sheet in REAL_SHEETS.glob("*.csv"):
        if sheet.stem in FOUR_SHEETS:
            shutil.copy(sheet, folder)
    cl.SHEETS, cl.OUT = folder, folder / "label_checks.json"
    return folder


def load(folder, sheet):
    path = folder / f"{sheet}.csv"
    frame = pd.read_csv(path, encoding="utf-8-sig", sep=";" if b";" in path.read_bytes()[:200] else ",")
    return frame.astype({frame.columns[-1]: "object"})


def fill(folder, sheet, column, pattern, rows=None, sep=",", encoding="utf-8-sig", name=None):
    """Write a fake label pattern by row position, never by reading the row."""
    frame = load(folder, sheet)
    count = len(frame) if rows is None else rows
    frame.loc[frame.index[:count], column] = [pattern[i % len(pattern)] for i in range(count)]
    frame.to_csv(folder / f"{name or sheet}.csv", index=False, sep=sep, encoding=encoding)


def fill_all(folder, second_labeler=True, **options):
    fill(folder, "challenge_precision", CHALLENGE, ["yes"] * 9 + ["no"], **options)
    fill(folder, "challenge_missed", CHALLENGE, ["no"], **options)
    fill(folder, "name_resolution", NAME, ["yes"], **options)
    fill(folder, "challenge_stance", STANCE, STANCE_PATTERN, **options)
    if second_labeler:
        fill(folder, "challenge_stance", STANCE, STANCE_PATTERN, name="challenge_stance_second", **options)


def stopped_with(function, *args):
    """The message of the SystemExit a call is meant to raise."""
    try:
        function(*args)
    except SystemExit as stop:
        return str(stop)
    raise AssertionError(f"{function.__name__} should have stopped")


def strict_json(path):
    """Parse JSON the way a browser does, where NaN is an error."""
    def reject(value):
        raise ValueError(f"{value} is not valid JSON")
    return json.loads(path.read_text(encoding="utf-8"), parse_constant=reject)


def run_main():
    with contextlib.redirect_stdout(io.StringIO()):
        cl.main()
    return strict_json(cl.OUT)


def outstanding(results):
    """The same test assets/site.js uses to decide whether to show the pending banner."""
    return [k for k in FOUR_SHEETS if results.get(k, {}).get("status") != "labeled"]


section = "setup"
check("check_labels imports inside a notebook", cl.main is not None)

PASS  check_labels imports inside a notebook


## 1. `wilson`: the confidence interval

Compared with scipy's own Wilson interval, which is separate code. scipy uses a slightly more exact z, so a small tolerance is allowed.

In [ ]:
section = "wilson"
for successes, total in [(54, 60), (60, 60), (0, 60), (1, 57)]:
    ours = cl.wilson(successes, total)
    interval = binomtest(successes, total).proportion_ci(method="wilson")
    check(f"{successes}/{total}: {ours} matches scipy", abs(ours[0] - interval.low) < 0.002 and abs(ours[1] - interval.high) < 0.002)
check("an empty sample gives None", cl.wilson(0, 0) is None)

PASS  54/60: [0.799, 0.953] matches scipy
PASS  60/60: [0.94, 1.0] matches scipy
PASS  0/60: [-0.0, 0.06] matches scipy
PASS  1/57: [0.003, 0.093] matches scipy
PASS  an empty sample gives None


## 2. `era` and `finite`: small helpers

In [3]:
section = "helpers"
cases = {"20091": "2009-2014", "20141": "2009-2014", "20142": "2014-2019", "20181": "2014-2019", "20191": "2019-2026", "20251": "2019-2026"}
check("era is right on both sides of each boundary", all(cl.era(s) == period for s, period in cases.items()))
check("era accepts a number as well as text", cl.era(20142) == cl.era("20142"))
sheet_eras = pd.read_csv(REAL_SHEETS / "challenge_precision.csv", encoding="utf-8-sig")["session"].map(cl.era).value_counts()
check(f"era gives roughly equal periods on the real sheet: {sheet_eras.to_dict()}", len(sheet_eras) == 3 and sheet_eras.max() - sheet_eras.min() <= 2)
check("finite rounds a normal number", cl.finite(0.12345, 3) == 0.123)
check("finite turns NaN and infinity into None", cl.finite(math.nan, 3) is None and cl.finite(math.inf, 3) is None)

PASS  era is right on both sides of each boundary
PASS  era accepts a number as well as text
PASS  era gives roughly equal periods on the real sheet: {'2019-2026': 21, '2014-2019': 20, '2009-2014': 19}
PASS  finite rounds a normal number
PASS  finite turns NaN and infinity into None


## 3. `read_sheet`: the ways a spreadsheet can save a file

In [4]:
section = "read_sheet"
folder = fresh()
fill(folder, "challenge_precision", CHALLENGE, ["yes"])
comma = cl.read_sheet(folder / "challenge_precision.csv")
fill(folder, "challenge_precision", CHALLENGE, ["yes"], sep=";")
semicolon = cl.read_sheet(folder / "challenge_precision.csv")
check("comma file reads as 60 rows and 10 columns", comma.shape == (60, 10))
check("semicolon file from Danish Excel reads the same as the comma file", comma.equals(semicolon))
check("Danish letters survive", comma["floor_ends"].str.contains("ø").any())

fill(folder, "challenge_precision", CHALLENGE, ["yes"], encoding="cp1252")
message = stopped_with(cl.read_sheet, folder / "challenge_precision.csv")
check(f"a non-UTF-8 file stops with: {message}", "not UTF-8" in message)

PASS  comma file reads as 60 rows and 10 columns
PASS  semicolon file from Danish Excel reads the same as the comma file
PASS  Danish letters survive
PASS  a non-UTF-8 file stops with: challenge_precision.csv is not UTF-8. Save it again as 'CSV UTF-8'.


## 4. `read_labels`: blanks, tidying and bad values

In [5]:
section = "read_labels"
folder = fresh()
check("a missing sheet gives (None, 0)", cl.read_labels("no_such_sheet", CHALLENGE, cl.YES_NO) == (None, 0))
labeled, total = cl.read_labels("challenge_precision", CHALLENGE, cl.YES_NO)
check("a blank sheet gives 0 labeled of 60", len(labeled) == 0 and total == 60)

fill(folder, "challenge_precision", CHALLENGE, ["  YES ", "No"], rows=10)
labeled, total = cl.read_labels("challenge_precision", CHALLENGE, cl.YES_NO)
check("spaces and capitals are tidied", set(labeled[CHALLENGE]) == {"yes", "no"})
check("10 of 60 rows counted as labeled", len(labeled) == 10 and total == 60)

for wrong in ["ja", "nej", "y", "n", "maybe"]:
    fill(folder, "challenge_precision", CHALLENGE, ["yes", wrong], rows=2)
    message = stopped_with(cl.read_labels, "challenge_precision", CHALLENGE, cl.YES_NO)
    check(f"{wrong!r} stops and names spreadsheet row 3", "row 3" in message and wrong in message)

fill(folder, "challenge_stance", STANCE, ["neutrl"], rows=1)
check("a stance typo stops the script", "neutrl" in stopped_with(cl.read_labels, "challenge_stance", STANCE, cl.STANCE_SCORE))
message = stopped_with(cl.read_labels, "challenge_precision", "renamed_column", cl.YES_NO)
check("a missing label column stops the script", "renamed_column" in message)

PASS  a missing sheet gives (None, 0)


PASS  a blank sheet gives 0 labeled of 60
PASS  spaces and capitals are tidied
PASS  10 of 60 rows counted as labeled
PASS  'ja' stops and names spreadsheet row 3
PASS  'nej' stops and names spreadsheet row 3


PASS  'y' stops and names spreadsheet row 3


PASS  'n' stops and names spreadsheet row 3
PASS  'maybe' stops and names spreadsheet row 3
PASS  a stance typo stops the script
PASS  a missing label column stops the script


## 5. `yes_share`: the `challenge_missed` score, and when a sheet counts as finished

In [6]:
section = "yes_share"
folder = fresh()
check("a blank sheet is not labeled", cl.yes_share("challenge_missed", CHALLENGE) == {"status": "not labeled yet"})

fill(folder, "challenge_missed", CHALLENGE, ["yes", "no", "no", "no"], rows=20)
result = cl.yes_share("challenge_missed", CHALLENGE)
check("20 of 60 rows is partial", result["status"] == "partial" and (result["n"], result["of"]) == (20, 60))
check("5 yes of 20 is 0.25", result["yes"] == 5 and result["share"] == 0.25)
check("the interval comes from wilson", result["ci95"] == cl.wilson(5, 20))

fill(folder, "challenge_missed", CHALLENGE, ["yes", "no", "no", "no"])
result = cl.yes_share("challenge_missed", CHALLENGE)
check("60 of 60 rows is labeled", result["status"] == "labeled" and result["yes"] == 15)

PASS  a blank sheet is not labeled


PASS  20 of 60 rows is partial
PASS  5 yes of 20 is 0.25
PASS  the interval comes from wilson


PASS  60 of 60 rows is labeled


## 6. `precision`: weighting by period

The fake labels say `yes` for every row in 2009-2014 and `no` otherwise, so the weighted share must equal that period's share of all edges. The expected value is counted straight from `challenges.parquet`, without using `cl.era`.

In [7]:
section = "precision"
folder = fresh()
check("a blank sheet is not labeled", cl.precision() == {"status": "not labeled yet"})

frame = load(folder, "challenge_precision")
frame[CHALLENGE] = ["yes" if str(s) < "20142" else "no" for s in frame["session"]]
frame.to_csv(folder / "challenge_precision.csv", index=False, encoding="utf-8-sig")
result = cl.precision()

sessions = pd.read_parquet(cl.DATA / "challenges.parquet", columns=["session"])["session"].astype(str)
expected = (sessions < "20142").mean()
check("by_era covers all 60 rows", sum(e["n"] for e in result["by_era"].values()) == 60)
check("by_era counts sum to the total yes", sum(e["yes"] for e in result["by_era"].values()) == result["yes"])
check(f"weighted share {result['era_weighted_share']} equals the edge share {expected:.3f}", result["era_weighted_share"] == round(expected, 3))
check("the plain share differs, because the sample is balanced by period", result["share"] != result["era_weighted_share"])

by_era_table = pd.DataFrame(result["by_era"]).T

folder = fresh()
fill(folder, "challenge_precision", CHALLENGE, ["yes"], rows=3)
result = cl.precision()
check("a partly filled sheet is partial and still gives a weighted share", result["status"] == "partial" and result["era_weighted_share"] == 1.0)
by_era_table

PASS  a blank sheet is not labeled


PASS  by_era covers all 60 rows
PASS  by_era counts sum to the total yes
PASS  weighted share 0.459 equals the edge share 0.459
PASS  the plain share differs, because the sample is balanced by period


PASS  a partly filled sheet is partial and still gives a weighted share


,n,yes
2009-2014,19,19
2014-2019,20,0
2019-2026,21,0


## 7. `names`: counting each name and person once

In [8]:
section = "names"
folder = fresh()
check("a blank sheet is not labeled", cl.names() == {"status": "not labeled yet"})

fill(folder, "name_resolution", NAME, ["yes"])
result = cl.names()
real = load(folder, "name_resolution")[["name_in_transcript", "matched_person_id"]].drop_duplicates()
check(f"57 rows are {len(real)} distinct pairs", result["n"] == 57 and result["distinct_pairs"] == len(real) == 33)

frame = load(folder, "name_resolution")
frame.loc[frame.index[frame["name_in_transcript"] == "Louise Schack Elholm"][0], NAME] = "no"
frame.to_csv(folder / "name_resolution.csv", index=False, encoding="utf-8-sig")
result = cl.names()
check("one no in a repeated pair fails that pair once", result["yes"] == 56 and result["distinct_yes"] == 32)
check("the distinct interval is on 33, not 57", result["distinct_ci95"] == cl.wilson(32, 33))

PASS  a blank sheet is not labeled
PASS  57 rows are 33 distinct pairs


PASS  one no in a repeated pair fails that pair once
PASS  the distinct interval is on 33, not 57


## 8. `stance`: both labelers, kappa and AFINN

Kappa and the AFINN correlation are recalculated here by separate code, from the formula and with pandas.

In [9]:
section = "stance"
folder = fresh()
check("a blank sheet is not labeled", cl.stance() == {"status": "not labeled yet"})

fill(folder, "challenge_stance", STANCE, STANCE_PATTERN)
result = cl.stance()
check("one labeler alone stays partial", result["status"] == "partial" and result["cohen_kappa"] == "no second labeler yet")
check("the label counts are right", result["distribution"] == {"neutral": 30, "hostile": 15, "friendly": 15})

first = load(folder, "challenge_stance")
afinn = Afinn(language="da")
scores = first["next_speaker_says"].map(afinn.score)
expected_rho = scores.corr(first[STANCE].map(cl.STANCE_SCORE), method="spearman")
check(f"AFINN correlation {result['afinn_spearman_with_labels']} matches pandas", abs(result["afinn_spearman_with_labels"] - expected_rho) < 0.001)

second = load(REAL_SHEETS, "challenge_stance")
second.loc[second.index[:5], STANCE] = first[STANCE].iloc[:5]
second.to_csv(folder / "challenge_stance_second.csv", index=False, encoding="utf-8-sig")
result = cl.stance()
check("a second labeler on 5 rows stays partial", result["status"] == "partial" and result["second_labeler_n"] == 5)

fill(folder, "challenge_stance", STANCE, STANCE_PATTERN, name="challenge_stance_second")
result = cl.stance()
check("full agreement is labeled with kappa 1", result["status"] == "labeled" and result["cohen_kappa"] == 1.0)

frame = load(folder, "challenge_stance")
frame.loc[frame.index[:15], STANCE] = "hostile"
frame.to_csv(folder / "challenge_stance_second.csv", index=False, encoding="utf-8-sig")
a, b = first[STANCE], frame[STANCE]
observed = (a == b).mean()
by_chance = sum((a == label).mean() * (b == label).mean() for label in cl.STANCE_SCORE)
expected_kappa = (observed - by_chance) / (1 - by_chance)
result = cl.stance()
check(f"kappa {result['cohen_kappa']} matches the formula {expected_kappa:.3f}", abs(result["cohen_kappa"] - expected_kappa) < 0.001)

fill(folder, "challenge_stance", STANCE, ["neutral"])
result = cl.stance()
check("all one label gives None, not NaN", result["afinn_spearman_with_labels"] is None and result["afinn_p"] is None)

PASS  a blank sheet is not labeled


PASS  one labeler alone stays partial
PASS  the label counts are right


PASS  AFINN correlation -0.108 matches pandas


PASS  a second labeler on 5 rows stays partial


PASS  full agreement is labeled with kappa 1


PASS  kappa 0.722 matches the formula 0.722


PASS  all one label gives None, not NaN


## 9. `main`: the whole run, and the site's banner

`main` is the function the pipeline runs. Each step below reads its output as strict JSON and applies the banner test from `assets/site.js`.

In [10]:
section = "main"
progress = []

folder = fresh()
progress.append(("blank sheets", outstanding(run_main())))
fill_all(folder, second_labeler=False, rows=1)
progress.append(("one row per sheet", outstanding(run_main())))
fill_all(folder, second_labeler=False)
progress.append(("all rows, one stance labeler", outstanding(run_main())))
fill_all(folder, second_labeler=True)
results = run_main()
progress.append(("all rows, two stance labelers", outstanding(results)))

display(pd.DataFrame([{"state": state, "sheets outstanding": len(left), "banner shown": bool(left)} for state, left in progress]))
check("blank sheets leave 4 outstanding", len(progress[0][1]) == 4)
check("one row per sheet does not clear the banner", len(progress[1][1]) == 4)
check("only challenge_stance is left without a second labeler", progress[2][1] == ["challenge_stance"])
check("all rows from both labelers clear the banner", progress[3][1] == [])

fill(folder, "challenge_stance", STANCE, ["neutral"])
check("the all-one-label case still writes valid JSON", run_main()["challenge_stance"]["afinn_spearman_with_labels"] is None)

folder = fresh()
fill_all(folder, sep=";")
semicolon_results = run_main()
fill_all(folder)
check("semicolon and comma files score the same", semicolon_results == run_main())

fill(folder, "challenge_missed", CHALLENGE, ["ja"])
check("a bad label stops main before it writes results", "ja" in stopped_with(cl.main))

,state,sheets outstanding,banner shown
0,blank sheets,4,True
1,one row per sheet,4,True
2,"all rows, one stance labeler",1,True
3,"all rows, two stance labelers",0,False


PASS  blank sheets leave 4 outstanding
PASS  one row per sheet does not clear the banner
PASS  only challenge_stance is left without a second labeler
PASS  all rows from both labelers clear the banner


PASS  the all-one-label case still writes valid JSON


PASS  semicolon and comma files score the same
PASS  a bad label stops main before it writes results


## 10. The real, unlabeled sheets

Run on the real sheets, `main` must report that nothing is labeled and match the committed results file. Its output still goes to the temporary folder.

In [11]:
section = "real sheets"
cl.SHEETS, cl.OUT = REAL_SHEETS, Path(workspace.name) / "real_sheets.json"
real = run_main()
print(json.dumps(real))
check("all four real sheets are not labeled yet", all(real[k] == {"status": "not labeled yet"} for k in FOUR_SHEETS))
check("the output matches label_checks.json in the repo", real == strict_json(REAL_OUT))

{"challenge_precision": {"status": "not labeled yet"}, "challenge_missed": {"status": "not labeled yet"}, "name_resolution": {"status": "not labeled yet"}, "challenge_stance": {"status": "not labeled yet"}}
PASS  all four real sheets are not labeled yet
PASS  the output matches label_checks.json in the repo


## Summary

This confirms the real files are unchanged, removes the temporary folder and lists every check.

In [12]:
section = "clean up"
cl.SHEETS, cl.OUT = REAL_SHEETS, REAL_OUT
check("the real sheets and label_checks.json are unchanged", fingerprint() == before)
workspace.cleanup()

table = pd.DataFrame(checks)
print(f"{(table['result'] == 'PASS').sum()} of {len(table)} checks passed")
table.groupby("section", sort=False)["result"].value_counts().unstack(fill_value=0)

PASS  the real sheets and label_checks.json are unchanged
59 of 59 checks passed


result,PASS
section,
setup,1
wilson,5
helpers,5
read_sheet,4
read_labels,11
yes_share,5
precision,6
names,4
stance,8
